In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, ed36c073-384b-4c7d-b10a-68fed764783f, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found")

StatementMeta(, ed36c073-384b-4c7d-b10a-68fed764783f, 4, Finished, Available, Finished, False)

16 projects found


In [4]:
all_vendors = []
page = 1

while True:
    response = requests.get(
        "https://api.procore.com/rest/v1.0/vendors",
        headers=headers,
        params={
            "company_id": COMPANY_ID,
            "page": page,
            "per_page": 100
        }
    )

    if response.status_code != 200:
        print(f"Error {response.status_code}")
        break

    rows = response.json()

    if not rows or isinstance(rows, dict):
        break

    all_vendors.extend(rows)

    if len(rows) < 100:
        break

    page += 1
    time.sleep(0.3)

print(f"Done! Total vendors: {len(all_vendors)}")

StatementMeta(, 21a57217-ed67-455f-a581-060a48b2505a, 6, Finished, Available, Finished, False)

Done! Total vendors: 1075


In [5]:
import pandas as pd
import re
import json

clean_rows = []
for row in all_vendors:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

# Drop and recreate to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS procore_vendors_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_vendors_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, 21a57217-ed67-455f-a581-060a48b2505a, 7, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
